In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [2]:
import sys, os

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
import project

add_path(os.environ['PROJECT'] + '/' + 'param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [3]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
data_root.is_dir()

True

In [4]:
import pandas as pd
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
len(subject_list)

1000

# Setup experiment

In [5]:
base_dir = '2026-09-15_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.subjects=[{subject}] \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.mesh_generation={mesh_tag} \\
    --set dataset.pipeline_tags.field_interpolation={int_tag} \\
    --set dataset.pipeline_tags.forward_simulation={sim_tag} \\
    --set preprocessing.mesh_generation.pygalmesh_kws.max_cell_circumradius={cell_radius} \\
    --set preprocessing.forward_simulation.pde_solver.material_type={material_type}

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-09-15_config.yaml',
    variant='2026-09-15',
    subject=subject_list,
    mesh_tag='rad',
    int_tag='fix', # no mask for u interpolation
    material_type=['LE', 'VK', 'NH', 'YH'],
    cell_radius=3.75
)

len(grid)

4000

In [12]:
base_dir = '2026-11-03_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.subjects=[{subject}] \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.mesh_generation={mesh_tag} \\
    --set dataset.pipeline_tags.field_interpolation={int_tag} \\
    --set dataset.pipeline_tags.forward_simulation={sim_tag} \\
    --set preprocessing.mesh_generation.pygalmesh_kws.max_cell_circumradius={cell_radius} \\
    --set preprocessing.forward_simulation.pde_solver.material_type={material_type} \\
    --set preprocessing.forward_simulation.pde_solver.incremental=true \\
    --set preprocessing.forward_simulation.pde_solver.verbose=true

'''
name_format = 'XN_{params_hash}'

grid = ps.param_grid(
    config='2026-09-15_config.yaml',
    variant='2026-09-15',
    subject=subject_list,
    mesh_tag='rad',
    int_tag='fix', # no mask for u interpolation
    material_type=['XN'],
    cell_radius=3.75
)

len(grid)

1000

In [13]:
radius_overrides = { # manually tested to resolve seg faults
    '17790S': 4.00,
    '19410S': 4.75,
    '13533E': 4.25,
    '17266B': 4.75,
}

bad_subjects = ['23686J', '24634V'] # mesh generation hangs

for p in grid:
    p['sim_tag'] = p['material_type']

    sid = p['subject']
    if sid in radius_overrides:
        p['cell_radius'] = radius_overrides[sid]


In [14]:
jobs = ps.setup(base_dir, template, name_format, grid, overwrite=True)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params_json,params_hash,params.config,params.variant,params.subject,params.mesh_tag,params.int_tag,params.material_type,params.cell_radius,params.sim_tag
0,job_74f7448bb977cfc8,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",74f7448bb977cfc8,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,16514P,rad,fix,XN,3.75,XN
1,job_e485ffe027f98110,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",e485ffe027f98110,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,20748Q,rad,fix,XN,3.75,XN
2,job_d1ffd742091a509e,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",d1ffd742091a509e,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,11007Z,rad,fix,XN,3.75,XN
3,job_6a417af520052bdf,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",6a417af520052bdf,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,14771Z,rad,fix,XN,3.75,XN
4,job_28bdfad61b3d7aa6,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",28bdfad61b3d7aa6,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,13651K,rad,fix,XN,3.75,XN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job_85e951f828837d08,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",85e951f828837d08,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,20519B,rad,fix,XN,3.75,XN
996,job_d2448c97bceb37a6,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",d2448c97bceb37a6,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,12294H,rad,fix,XN,3.75,XN
997,job_38c42d0c4b2304c4,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",38c42d0c4b2304c4,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,23123R,rad,fix,XN,3.75,XN
998,job_a4980a19e0fb959e,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""cell_radius"": 3.75, ""config"": ""2026-09-15_co...",a4980a19e0fb959e,/projectnb/batmanlab/mragoza/lung-project/note...,2026-09-15,16546C,rad,fix,XN,3.75,XN


In [ ]:
#jobs.loc[:, 'job_id'] = pd.NA
#jobs = ps.submit(jobs)

# Monitor submitted jobs

In [5]:
jobs = ps.load(base_dir)
jobs

NameError: name 'base_dir' is not defined

In [8]:
%%time
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs) # not working for SCC cluster; job state may be stale
jobs = ps.collect(jobs)

jobs.groupby('job_state').job_name.describe()

CPU times: user 1.08 s, sys: 2.02 s, total: 3.1 s
Wall time: 2min 34s


,count,unique,top,freq
job_state,,,,
PENDING,2767,2767,job_195a5dfdb0441a93,1
RUNNING,27,27,job_c40b80191f8e9046,1
SUBMITTED,1206,1206,job_c7007414048315fb,1


In [9]:
def classify_output(stdout):
    if pd.isnull(stdout):
        return 'No stdout file'
    elif not str(stdout).strip():
        return 'No output message'
    elif 'Done' in stdout:
        return 'Complete'
    return 'Incomplete'

def classify_error(stderr):
    if pd.isnull(stderr):
        return 'No stderr file'  
    elif not str(stderr).strip():
        return 'No error message'
    elif 'Segmentation fault' in stderr:
        return 'Segmentation fault'
    elif 'CUDA out of memory' in stderr:
        return 'CUDA out of memory'
    elif 'CUDA error: uncorrectable ECC error' in stderr:
        return 'CUDA ECC error'
    elif 'No CUDA GPUs are available' in stderr:
        return 'GPU not available'
    elif 'Newton solver failed to converge' in stderr:
        return 'Failed to converge'
    elif 'Non-finite PDE residual' in stderr:
        return 'NaN PDE residual'
    elif 'Non-finite GMRES residual' in stderr:
        return 'NaN GMRES residual'
    elif 'OSError' in stderr:
        return 'OSError'
    elif 'WARNING: Interpolation points outside domain' in stderr:
        return 'Interpolation warning'
    return 'Uncategorized'

jobs['output_type'] = jobs.stdout.map(classify_output)
jobs['error_type'] = jobs.stderr.map(classify_error)

jobs.groupby(['params.material_type', 'output_type', 'error_type'])[['job_name']].agg(['count', 'unique'])

job_name  \
                                                          count   
params.material_type output_type error_type                       
LE                   Complete    Interpolation warning      878   
                                 No error message           120   
                     Incomplete  No error message             2   
NH                   Complete    Failed to converge         137   
                                 Interpolation warning      284   
                                 No error message           577   
                     Incomplete  No error message             2   
VK                   Complete    Failed to converge          99   
                                 Interpolation warning      425   
                                 No error message           471   
                     Incomplete  NaN GMRES residual           1   
                                 NaN PDE residual             2   
                                 No error message             2   
YH                   Complete    Failed to converge           5   
                                 Interpolation warning      149   
                                 No error message           517   
                     Incomplete  GPU not available            2   
                                 NaN PDE residual           325   
                                 No error message             2   

                                                                                                           
                                                                                                   unique  
params.material_type output_type error_type                                                                
LE                   Complete    Interpolation warning  [job_70bd7f922217da55, job_7f989a49a0de8f4a, j...  
                                 No error message       [job_c7007414048315fb, job_c40b80191f8e9046, j...  
                     Incomplete  No error message            [job_d2e2b3fba5cb1fef, job_ee978a80a0c67008]  
NH                   Complete    Failed to converge     [job_e397c48609bd8f15, job_b7037684f0277b64, j...  
                                 Interpolation warning  [job_45fa8076eae906fe, job_9c631a5cd9160b4b, j...  
                                 No error message       [job_eb1073034b6519dd, job_8ee4d6e057cfc7b6, j...  
                     Incomplete  No error message            [job_a865c6f8f766d4e0, job_77833513d6c264af]  
VK                   Complete    Failed to converge     [job_195a5dfdb0441a93, job_f08e488e4c4a179a, j...  
                                 Interpolation warning  [job_f44ead389a836ecf, job_d4c41762584094df, j...  
                                 No error message       [job_7aaac7550550bbdd, job_d05b1fc649611eec, j...  
                     Incomplete  NaN GMRES residual                                [job_683019f5c3ce3eea]  
                                 NaN PDE residual            [job_403f5da73d201180, job_ce6a1158c5b89cfa]  
                                 No error message            [job_20fc30e1707313f8, job_a1636bf1600f21f4]  
YH                   Complete    Failed to converge     [job_2648e6f3604b5acc, job_eeb23a49303d188a, j...  
                                 Interpolation warning  [job_bdb5796e97ca2df1, job_c1e0cd477761ba35, j...  
                                 No error message       [job_80c7027586be0936, job_53179f0d9af5ee98, j...  
                     Incomplete  GPU not available           [job_7ef2cae9c69e21fe, job_05ccd75c2b6bc151]  
                                 NaN PDE residual       [job_505e051f3d12f50f, job_7b337935d1cec5c7, j...  
                                 No error message            [job_1ddf654079695af1, job_217acc8b521696dd]

In [10]:
latest_job_ids = jobs.groupby(['job_name']).job_id.max()
latest_jobs = jobs[jobs.job_id.isin(latest_job_ids)]

latest_jobs.groupby(['params.material_type', 'output_type', 'error_type', ])[['job_name']].count()

job_name
params.material_type output_type error_type                     
LE                   Complete    Interpolation warning       878
                                 No error message            120
                     Incomplete  No error message              2
NH                   Complete    Failed to converge          137
                                 Interpolation warning       284
                                 No error message            577
                     Incomplete  No error message              2
VK                   Complete    Failed to converge           99
                                 Interpolation warning       425
                                 No error message            471
                     Incomplete  NaN GMRES residual            1
                                 NaN PDE residual              2
                                 No error message              2
YH                   Complete    Failed to converge            5
                                 Interpolation warning       149
                                 No error message            517
                     Incomplete  GPU not available             2
                                 NaN PDE residual            325
                                 No error message              2

In [11]:
latest_jobs.groupby(['params.material_type', 'output_type'])[['job_name']].count()

job_name
params.material_type output_type          
LE                   Complete          998
                     Incomplete          2
NH                   Complete          998
                     Incomplete          2
VK                   Complete          995
                     Incomplete          5
YH                   Complete          671
                     Incomplete        329

In [12]:
query_jobs = jobs.query('error_type == "Uncategorized"')
query_jobs.groupby(['params.material_type']).describe()

Empty DataFrame
Columns: [(n_submits, count), (n_submits, mean), (n_submits, std), (n_submits, min), (n_submits, 25%), (n_submits, 50%), (n_submits, 75%), (n_submits, max), (params.cell_radius, count), (params.cell_radius, mean), (params.cell_radius, std), (params.cell_radius, min), (params.cell_radius, 25%), (params.cell_radius, 50%), (params.cell_radius, 75%), (params.cell_radius, max), (array_idx, count), (array_idx, mean), (array_idx, std), (array_idx, min), (array_idx, 25%), (array_idx, 50%), (array_idx, 75%), (array_idx, max)]
Index: []

[0 rows x 24 columns]

In [13]:
job = query_jobs.iloc[0]
print(job)
print(job.stdout)
print(job.stderr)

IndexError: single positional indexer is out-of-bounds

# Resubmit selected jobs

In [23]:
jobs.loc[jobs.error_type == 'Failed to converge', 'job_id'] = pd.NA
jobs = ps.submit(jobs)

# Validate simulation results

In [11]:
mesh_cache = {}

def load_mesh_with_cache(path):
    key = str(path)
    if key not in mesh_cache:
        mesh_cache[key] = project.fileio.load_meshio(path)
    return mesh_cache[key]


In [47]:
import numpy as np
import project.common.transforms

def validate_simulation(path, key='u_fwd'):
    mesh = load_mesh_with_cache(path)

    cells = mesh.cells_dict.get('tetra', [])
    X = np.asarray(mesh.points, dtype=np.float64)
    u = np.asarray(mesh.point_data[key], dtype=np.float64)

    N, M = len(X), len(cells)
    
    if N == 0:
        raise ValueError(f'No vertices')
    if M == 0:
        raise ValueError(f'No tetrahedral cells')
    if X.shape != (N, 3):
        raise ValueError(f'Invalid coordinates shape: {X.shape} vs. {(N, 3)}')
    if u.shape != (N, 3):
        raise ValueError(f'Invalid displacement shape: {u.shape} vs. {(N, 3)}')
    if cells.shape != (M, 4):
        raise ValueError(f'Invalid tetra cells shape: {cells.shape} vs. {(M, 4)}')
    if not np.all(np.isfinite(X)):
        raise ValueError(f'Non-finite coordinates'
    if not np.all(np.isfinite(u)):
        raise ValueError(f'Non-finite displacement')
    if not np.all((0 <= cells) & (cells < N)):
        raise ValueError(f'Invalid tetra cell indices')

    x = X + u

    P = X[cells] # (M, 4, 3)
    p = x[cells]

    V0 = np.linalg.det(P[:,1:] - P[:,:1]) / 6
    V1 = np.linalg.det(p[:,1:] - p[:,:1]) / 6

    if not np.all(np.isfinite(V0)):
        raise ValueError('Non-finite reference volume(s)')

    if np.any(V0 == 0):
        raise ValueError('Degenerate reference tetrahedra')

    J = V1 /  V0

    inverted = (J <= 0)
    nonfinite = ~np.isfinite(J)
    bad = inverted | nonfinite
    finite_J = J[np.isfinite(J)]

    summary = {
        'status': 'FAIL' if bad.any() else 'PASS',
        'n_tets': len(cells),
        'min_J': finite_J.min() if finite_J.size > 0 else np.nan,
        'n_inverted': int(inverted.sum()),
        'n_nonfinite': int(nonfinite.sum()),
        'p_inverted': float(inverted.mean()),
        'p_nonfinite': float(nonfinite.mean())
    }

    details = []

    if not np.any(bad):
        return summary, details

    inds = np.flatnonzero(bad)
    bad_P = P[inds]

    edges2 = sum(
        np.sum((bad_P[:,i] - bad_P[:,j])**2, axis=1)
            for i in range(4) for j in range(i)
    )
    quality = 12 * (3 * np.abs(V0[inds]))**(2/3) / edges2
    
    b = np.asarray(
        project.common.transforms.get_surface_mask(X, cells),
        dtype=bool
    ).reshape(-1)

    for i, q in zip(inds, quality):
        center = P[i].mean(axis=0)
        details.append({
            'cell': int(i),
            'J': float(J[i]),
            'ref_quality': float(q),
            'n_boundary_verts': int(b[cells[i]].sum()),
            'center_x': center[0],
            'center_y': center[1],
            'center_z': center[2]
        })

    return summary, details


In [26]:
import pandas as pd
from tqdm.auto import tqdm

def validate_deformation(ex):
    from project.common import transforms
    ret = {'subject': ex.subject}

    disp_path = ex.paths.get('disp_field')
    nifti = project.fileio.load_nibabel(disp_path)
    disp, affine = nifti.get_fdata(), nifti.affine

    J = transforms.voxel_deformation_jacobian(disp, affine)
    bad = (J <= 0)

    ret['u_vox.min_J'] = J.min()
    ret['u_vox.n_bad'] = bad.sum()
    ret['u_vox.p_bad'] = bad.mean()

    mesh_path = ex.paths.get('target_mesh')
    mesh = project.fileio.load_meshio(mesh_path)

    J = transforms.mesh_deformation_jacobian(mesh, key='u_reg')
    bad = (J <= 0)

    ret['u_reg.min_J'] = J.min()
    ret['u_reg.n_bad'] = bad.sum()
    ret['u_reg.p_bad'] = bad.mean()

    J = transforms.mesh_deformation_jacobian(mesh, key='u_fwd')
    bad = (J <= 0)

    ret['u_fwd.min_J'] = J.min()
    ret['u_fwd.n_bad'] = bad.sum()
    ret['u_fwd.p_bad'] = bad.mean()

    return ret

project.utils.set_verbose(True)

config = project.fileio.load_config('2026-09-15_config.yaml')
config['dataset']['variant'] = '2026-09-15'
config['dataset']['pipeline_tags']['mesh_generation'] = 'rad'
config['dataset']['pipeline_tags']['field_interpolation'] = 'fix'
config['dataset']['pipeline_tags']['forward_simulation'] = 'LE'

examples = project.api.get_examples(config['dataset'])[:1]

rows = []
for ex in tqdm(examples, desc='Validating deformations'):
    rows.append(validate_deformation(ex))

df = pd.DataFrame(rows)
df

Loading 2026-09-15_config.yaml
Gathering examples...
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.csv
1000 total


Validating deformations:   0%|          | 0/1 [00:00<?, ?it/s]

Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-09-15/16514P/fields/16514P_EXP_iso_v3d_gil_INSP.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-09-15/16514P/meshes/16514P_EXP_iso_v3d_gil_INSP_lung_rad_bela_int_LE.xdmf


,subject,u_vox.min_J,u_vox.n_bad,u_vox.p_bad,u_reg.min_J,u_reg.n_bad,u_reg.p_bad,u_fwd.min_J,u_fwd.n_bad,u_fwd.p_bad
0,16514P,0.311965,0,0.0,0.357007,0,0.0,-0.091453,1,0.000006


In [54]:
ex_df[ex_df.status == 'ERROR']

,subject,path,status,n_tets,min_J,n_inverted,n_nonfinite,p_inverted,p_nonfinite,error
445,23686J,/restricted/projectnb/batmanlab/mragoza/data/C...,ERROR,NaN,NaN,NaN,NaN,NaN,NaN,File /restricted/projectnb/batmanlab/mragoza/d...
873,24634V,/restricted/projectnb/batmanlab/mragoza/data/C...,ERROR,NaN,NaN,NaN,NaN,NaN,NaN,File /restricted/projectnb/batmanlab/mragoza/d...


In [55]:
error_sids = [examples[i].subject for i in ex_df[ex_df.status == 'ERROR'].index]
error_sids

['23686J', '24634V']

In [56]:
jobs[jobs['params.subject'].isin(error_sids) & (jobs['params.material_type'] == 'LE')]

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.material_type,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime,error_type
1780,job_bf491a631e81510a,PENDING,1,7698702,None,None,| ├── 'type': 'AdamW'\n | └── 'lr':...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,NaN,2026-09-22T22:25:43,status,None,None,False,<NA>,<NA>,No error message
3492,job_16ebd19742609737,PENDING,1,7700419,None,None,| ├── 'type': 'AdamW'\n | └── 'lr':...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,LE,NaN,2026-09-22T22:25:43,status,None,None,False,<NA>,<NA>,No error message


In [57]:
ex_df[ex_df.status == 'FAIL']

,subject,path,status,n_tets,min_J,n_inverted,n_nonfinite,p_inverted,p_nonfinite,error
5,15900P,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,171343.0,-0.081871,1.0,0.0,0.000006,0.0,NaN
15,10572Z,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,294804.0,-0.080854,1.0,0.0,0.000003,0.0,NaN
54,24608U,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,179482.0,-0.030111,1.0,0.0,0.000006,0.0,NaN
96,17104B,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,236916.0,-0.224034,1.0,0.0,0.000004,0.0,NaN
99,13170U,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,229398.0,-0.038891,1.0,0.0,0.000004,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...
962,20818L,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,405103.0,-0.147678,1.0,0.0,0.000002,0.0,NaN
969,15140V,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,319092.0,-0.160322,1.0,0.0,0.000003,0.0,NaN
978,13404T,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,368054.0,-0.162293,1.0,0.0,0.000003,0.0,NaN
987,13065X,/restricted/projectnb/batmanlab/mragoza/data/C...,FAIL,212368.0,-0.003180,1.0,0.0,0.000005,0.0,NaN


In [60]:
ex_df.groupby('status').describe().stack()

/scratch/7897589.1.engineering/ipykernel_2794755/420891087.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  ex_df.groupby('status').describe().stack()


n_tets       min_J  n_inverted  n_nonfinite  p_inverted  \
status                                                                         
ERROR  count       0.000000    0.000000    0.000000          0.0    0.000000   
FAIL   count      81.000000   81.000000   81.000000         81.0   81.000000   
       mean   331945.666667   -0.229555    1.098765          0.0    0.000004   
       std     87386.697681    0.307213    0.339298          0.0    0.000001   
       min    171343.000000   -1.998906    1.000000          0.0    0.000002   
       25%    272711.000000   -0.305180    1.000000          0.0    0.000003   
       50%    319092.000000   -0.123241    1.000000          0.0    0.000003   
       75%    403001.000000   -0.047746    1.000000          0.0    0.000004   
       max    493879.000000   -0.002236    3.000000          0.0    0.000007   
PASS   count     917.000000  917.000000  917.000000        917.0  917.000000   
       mean   312996.244275    0.314772    0.000000          0.0    0.000000   
       std     94342.497527    0.134452    0.000000          0.0    0.000000   
       min      2639.000000    0.002596    0.000000          0.0    0.000000   
       25%    239565.000000    0.231358    0.000000          0.0    0.000000   
       50%    300982.000000    0.316291    0.000000          0.0    0.000000   
       75%    375396.000000    0.401624    0.000000          0.0    0.000000   
       max    499996.000000    0.880894    0.000000          0.0    0.000000   

              p_nonfinite  
status                     
ERROR  count          0.0  
FAIL   count         81.0  
       mean           0.0  
       std            0.0  
       min            0.0  
       25%            0.0  
       50%            0.0  
       75%            0.0  
       max            0.0  
PASS   count        917.0  
       mean           0.0  
       std            0.0  
       min            0.0  
       25%            0.0  
       50%            0.0  
       75%            0.0  
       max            0.0

In [61]:
tet_df.describe()

,cell,J,ref_quality,n_boundary_verts,center_x,center_y,center_z
count,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000,89.000000
mean,164309.247191,-0.214250,0.373244,3.460674,-3.183387,105.603962,-73.354418
std,104397.392127,0.297310,0.096829,1.206575,80.373579,100.674820,432.075984
min,6373.000000,-1.998906,0.097109,0.000000,-143.090187,-88.701588,-1012.322266
25%,96220.000000,-0.276737,0.321734,4.000000,-63.705139,4.590683,-243.186970
50%,148551.000000,-0.108522,0.368487,4.000000,-1.098106,128.651051,-177.651012
75%,219466.000000,-0.039903,0.410992,4.000000,69.141016,177.989151,-98.845108
max,447992.000000,-0.001346,0.630255,4.000000,142.595818,271.384132,1301.978962


In [62]:
print(tet_df['n_boundary_verts'].value_counts().sort_index())

fixed = tet_df['n_boundary_verts'].eq(4)
print("Cases with fully constrained inverted tets:",
      tet_df.loc[fixed, 'subject'].nunique())

n_boundary_verts
0     5
1     6
2     5
4    73
Name: count, dtype: int64
Cases with fully constrained inverted tets: 69


In [65]:
tet_df.loc[fixed, 'subject'].unique()

array(['15900P', '24608U', '17104B', '13170U', '21052K', '24271L',
       '14217Z', '21587X', '25096Y', '20623W', '20678V', '17144N',
       '15350G', '16700M', '10736D', '20867Y', '19024N', '21065T',
       '23743V', '19730K', '21604X', '13945B', '23741R', '21622Z',
       '10124Y', '24646C', '10880K', '24154H', '10705S', '10130T',
       '13533E', '19921R', '14201K', '25309P', '21293G', '16266W',
       '17894E', '21640B', '10568I', '11353S', '11556G', '11027F',
       '13267J', '24927K', '10081G', '15606P', '14307A', '15885T',
       '25003R', '24350H', '18786G', '22148C', '10552T', '16954R',
       '24989G', '14964K', '14257L', '22832P', '23312U', '10114V',
       '14909C', '11815E', '23615M', '20349C', '17476M', '20818L',
       '13404T', '13065X', '14754Z'], dtype=object)

In [102]:
bad_subjects = ['15900P', '24608U', '17104B', '13170U', '21052K', '24271L',
       '14217Z', '21587X', '25096Y', '20623W', '20678V', '17144N',
       '15350G', '16700M', '10736D', '20867Y', '19024N', '21065T',
       '23743V', '19730K', '21604X', '13945B', '23741R', '21622Z',
       '10124Y', '24646C', '10880K', '24154H', '10705S', '10130T',
       '13533E', '19921R', '14201K', '25309P', '21293G', '16266W',
       '17894E', '21640B', '10568I', '11353S', '11556G', '11027F',
       '13267J', '24927K', '10081G', '15606P', '14307A', '15885T',
       '25003R', '24350H', '18786G', '22148C', '10552T', '16954R',
       '24989G', '14964K', '14257L', '22832P', '23312U', '10114V',
       '14909C', '11815E', '23615M', '20349C', '17476M', '20818L',
       '13404T', '13065X', '14754Z']

bad_examples = [x for x in examples if x.subject in bad_subjects]

In [84]:
m.shape

(320, 320, 283)

In [105]:
def validate_registration(disp_path, mask_path):

    disp_nifti = project.fileio.load_nibabel(disp_path)
    mask_nifti = project.fileio.load_nibabel(mask_path)
    
    u = disp_nifti.get_fdata()
    m = mask_nifti.get_fdata() > 0

    if not u.ndim == 4 or u.shape[-1] != 3:
        raise ValueError(f'Invalid deformation shape: {u.shape}')

    if m.shape != u.shape[:-1]:
        raise ValueError(f'Invalid mask shape: {m.shape} vs. {u.shape[:-1]}')

    if not np.allclose(disp_nifti.affine, mask_nifti.affine):
        raise ValueError(f'Displacement and mask grids differ')

    if not m.any():
        raise ValueError('Empty mask')

    grad_voxel = np.empty(m.shape + (3,3), dtype=np.float64)
    
    for component in range(3):
        derivatives = np.gradient(u[...,component], edge_order=2)
        for axis in range(3):
            grad_voxel[..., component, axis] = derivatives[axis]

    grad_world = grad_voxel @ np.linalg.inv(disp_nifti.affine[:3,:3])
    
    F = grad_world + np.eye(3)
    J = np.linalg.det(F)[m]

    nonpositive = J <= 0
    nonfinite = ~np.isfinite(J)
    bad_disp = ~np.isfinite(u[m]).all(axis=-1)
    finite_J = J[np.isfinite(J)]

    if nonpositive.any() or nonfinite.any() or bad_disp.any():
        status = 'FAIL'
    else:
        status = 'PASS'

    summary = {
        'status': status,
        'n_voxels': int(m.sum()),
        'min_J': float(finite_J.min()) if finite_J.size > 0 else np.nan,
        'n_nonpositive': int(nonpositive.sum()),
        'p_nonpositive': float(nonpositive.mean()),
        'n_nonfinite': int(nonfinite.sum()),
        'p_nonfinite': float(nonfinite.mean()),
        'n_bad_disp': int(bad_disp.sum())
    }
    return summary
    

ex = next(x for x in examples if x.subject == '15900P')

disp_path = ex.paths['disp_field']
mask_path = ex.paths['anatomical_map']

validate_registration(disp_path, mask_path)

{'status': 'PASS',
 'n_voxels': 2051985,
 'min_J': 0.45040738921946555,
 'n_nonpositive': 0,
 'p_nonpositive': 0.0,
 'n_nonfinite': 0,
 'p_nonfinite': 0.0,
 'n_bad_disp': 0}

In [106]:
ex_rows = []

for ex in tqdm(bad_examples, desc='Validating registration'):
    disp_path = ex.paths['disp_field']
    mask_path = ex.paths['anatomical_map']
    base = {'subject': ex.subject, 'path': str(disp_path)}

    try:
        summary = validate_registration(disp_path, mask_path)
    except Exception as e:
        ex_rows.append({**base, 'status': 'ERROR', 'error': str(e)})
    else:
        ex_rows.append({**base, **summary})

ex_df = pd.DataFrame(ex_rows)

Validating registration:   0%|          | 0/69 [00:00<?, ?it/s]

In [117]:
ex_df

,subject,path,status,n_voxels,min_J,n_nonpositive,p_nonpositive,n_nonfinite,p_nonfinite,n_bad_disp
0,15900P,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,2051985,0.450407,0,0.0,0,0.0,0
1,24608U,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,2137266,0.334888,0,0.0,0,0.0,0
2,17104B,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,2853852,0.154285,0,0.0,0,0.0,0
3,13170U,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,2765090,0.209527,0,0.0,0,0.0,0
4,21052K,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,2135703,0.242521,0,0.0,0,0.0,0
...,...,...,...,...,...,...,...,...,...,...
64,17476M,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,4876179,0.297064,0,0.0,0,0.0,0
65,20818L,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,4928986,0.318223,0,0.0,0,0.0,0
66,13404T,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,4444478,0.381463,0,0.0,0,0.0,0
67,13065X,/restricted/projectnb/batmanlab/mragoza/data/C...,PASS,2549249,0.227715,0,0.0,0,0.0,0


In [118]:
ex_df = pd.DataFrame(ex_rows)
ex_df.groupby('status').describe().stack()

/scratch/7897589.1.engineering/ipykernel_2794755/390477590.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  ex_df.groupby('status').describe().stack()


n_voxels      min_J  n_nonpositive  p_nonpositive  \
status                                                                
PASS   count  6.900000e+01  69.000000           69.0           69.0   
       mean   4.098187e+06   0.310639            0.0            0.0   
       std    1.244407e+06   0.084195            0.0            0.0   
       min    2.051985e+06   0.116600            0.0            0.0   
       25%    3.265024e+06   0.251360            0.0            0.0   
       50%    3.853649e+06   0.303415            0.0            0.0   
       75%    4.928986e+06   0.364853            0.0            0.0   
       max    7.518863e+06   0.574419            0.0            0.0   

              n_nonfinite  p_nonfinite  n_bad_disp  
status                                              
PASS   count         69.0         69.0        69.0  
       mean           0.0          0.0         0.0  
       std            0.0          0.0         0.0  
       min            0.0          0.0         0.0  
       25%            0.0          0.0         0.0  
       50%            0.0          0.0         0.0  
       75%            0.0          0.0         0.0  
       max            0.0          0.0         0.0

In [129]:
def voxel_deformation_jacobian(disp, affine):

    if disp.ndim != 4 or disp.shape[-1] != 3:
        raise ValueError(f'Invalid displacement shape: {disp.shape}')

    grad_v = np.empty(disp.shape[:3] + (3, 3), dtype=np.float64)

    for i in range(3):
        for j, deriv in enumerate(np.gradient(disp[...,i], edge_order=2)):
            grad_v[...,i,j] = deriv

    grad_w = grad_v @ np.linalg.inv(affine[:3,:3])

    return np.linalg.det(grad_w + np.eye(3))


def mesh_deformation_jacobian(mesh, key):
    
    cells = mesh.cells_dict.get('tetra', [])
    if len(cells) == 0:
        raise ValueError('Mesh has no tetrahedral cells')
    
    x_ref = np.asarray(mesh.points, dtype=np.float64)
    u = np.asarray(mesh.point_data[key], dtype=np.float64)

    x_def = x_ref + u
    p_ref = x_ref[cells]
    p_def = x_def[cells]

    V_ref = np.linalg.det(p_ref[:,1:] - p_ref[:,:1]) / 6

    if not np.isfinite(V_ref).all() or np.any(V_ref == 0):
        raise ValueError('Invalid reference tetrahedra')

    V_def = np.linalg.det(p_def[:,1:] - p_def[:,:1]) / 6

    return V_def / V_ref


def test_interpolation(disp_path, mask_path, mesh_path):
    from project.common import fileio, transforms, interpolation

    disp_nifti = fileio.load_nibabel(disp_path)
    mask_nifti = fileio.load_nibabel(mask_path)
    mesh = fileio.load_meshio(mesh_path)

    disp = disp_nifti.get_fdata()
    mask = mask_nifti.get_fdata() > 0
    affine = disp_nifti.affine

    if mask.shape != disp.shape[:3]:
        raise ValueError('Mask/displacement shape mismatch')

    if not np.allclose(mask_nifti.affine, affine, rtol=0, atol=1e-5):
        raise ValueError('Mask/displacement affines differ')

    J_voxel = voxel_deformation_jacobian(disp, affine)

    node_voxels = transforms.world_to_voxel_coords(mesh.points, affine)

    mesh.point_data['u_direct'] = interpolation.interpolate_array(disp, node_voxels)
    mesh.point_data['u_masked'] = interpolation.interpolate_masked(disp, mask, node_voxels)

    J_direct = mesh_deformation_jacobian(mesh, key='u_direct')
    J_masked = mesh_deformation_jacobian(mesh, key='u_masked')

    mesh.cell_data['J_direct'] = [J_direct]
    mesh.cell_data['J_masked'] = [J_masked]

    def summarize(J):
        J_finite = J[np.isfinite(J)]
        return {
            'min_J': float(J_finite.min()) if J_finite.size > 0 else np.nan,
            'nonpositive': int((J <= 0).sum()),
            'nonfinite': int((~np.isfinite(J)).sum())
        }

    summary = {
        'voxel': summarize(J_voxel[mask]),
        'direct': summarize(J_direct),
        'masked': summarize(J_masked)
    }
    return mesh, summary

mesh_path = ex.paths['target_mesh']
mask_path = ex.paths['anatomical_map']
disp_path = ex.paths['disp_field']

test_interpolation(disp_path, mask_path, mesh_path)

(<meshio mesh object>
   Number of points: 34426
   Number of cells:
     tetra: 171343
   Point data: medit:ref, image, u_reg, E, nu, rho, u_fwd, u_direct, u_masked
   Cell data: medit:ref, label, image, u_reg, E, nu, rho, u_fwd, J_direct, J_masked,
 {'voxel': {'min_J': 0.45040738921946555, 'nonpositive': 0, 'nonfinite': 0},
  'direct': {'min_J': -0.09086211514519597, 'nonpositive': 1, 'nonfinite': 0},
  'masked': {'min_J': -0.0818709852545324, 'nonpositive': 1, 'nonfinite': 0}})

In [130]:
ex_rows = []

for ex in tqdm(bad_examples, desc='Validating interpolation'):
    
    disp_path = ex.paths['disp_field']
    mask_path = ex.paths['anatomical_map']
    mesh_path = ex.paths['target_mesh']

    mesh, summary = test_interpolation(disp_path, mask_path, mesh_path)

    ex_rows.append({
        'subject': ex.subject,
        
        'voxel_min_J': summary['voxel']['min_J'],
        'direct_min_J': summary['direct']['min_J'],
        'masked_min_J': summary['masked']['min_J'],

        'voxel_nonpositive': summary['voxel']['nonpositive'],
        'direct_nonpositive': summary['direct']['nonpositive'],
        'masked_nonpositive': summary['masked']['nonpositive'],

        'voxel_nonfinite': summary['voxel']['nonfinite'],
        'direct_nonfinite': summary['direct']['nonfinite'],
        'masked_nonfinite': summary['masked']['nonfinite'],
    })

ex_df = pd.DataFrame(ex_rows)

Validating interpolation:   0%|          | 0/69 [00:00<?, ?it/s]